# Fixed & Optimized SwinUNETR BraTS Training
### Fixes applied:
- ✅ `labels.squeeze(1)` removed — correct shape for DiceCELoss
- ✅ Sliding window inference for validation (was doing full-volume forward pass → wrong results + OOM)
- ✅ `decollate_batch` added for correct post_pred / post_label application
- ✅ `ConvertLabels` uses torch instead of numpy (MetaTensor-safe)
- ✅ BATCH_SIZE increased to 4 (was leaving ~40GB VRAM idle on A6000)
- ✅ `use_checkpoint=True` on SwinUNETR (saves ~30% VRAM)
- ✅ NUM_WORKERS corrected to 8 (Xeon Bronze 3106 = 8 cores/socket, no HT)
- ✅ val cache_rate lowered to 0.5 to avoid RAM exhaustion
- ✅ SW_BATCH_SIZE increased to 4 for faster sliding window
- ✅ Cosine annealing LR scheduler added
- ✅ VAL_INTERVAL set to 2 (every epoch was too expensive)
- ✅ `cudnn.benchmark = True` for cuDNN auto-tuning
- ✅ `include_background=False`, `reduction='mean_batch'` for correct BraTS Dice (WT/TC/ET)
- ✅ Best model checkpoint saving added
- ✅ Removed unused FOLDS variable

In [1]:
from pathlib import Path
import torch

# ── cuDNN auto-tune: finds fastest conv algorithm for fixed input size ──
torch.backends.cudnn.benchmark = True

DATA_ROOT = Path(r"C:\Users\Admin\Desktop\meghana\Brats2024\BraTS2024-BraTS-GLI-TrainingData\training_data1_v2")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ROI_SIZE      = (128, 128, 128)
BATCH_SIZE    = 4          # FIX: was 1 — A6000 has 48GB VRAM, use it
NUM_WORKERS   = 8          # FIX: was 12 — Xeon Bronze 3106 = 8 cores/socket, no hyperthreading
MAX_EPOCHS    = 20
LR            = 1e-4
VAL_INTERVAL  = 2          # FIX: was 1 — sliding window val over 270 full volumes is expensive
SW_BATCH_SIZE = 4          # FIX: was 1 — process 4 windows in parallel on A6000

NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA RTX A6000
VRAM: 51.5 GB


In [2]:
train_files = []

for case in DATA_ROOT.iterdir():
    if case.is_dir():
        case_name = case.name

        t1    = case / f"{case_name}-t1n.nii.gz"
        t1ce  = case / f"{case_name}-t1c.nii.gz"
        t2    = case / f"{case_name}-t2w.nii.gz"
        flair = case / f"{case_name}-t2f.nii.gz"
        seg   = case / f"{case_name}-seg.nii.gz"

        if all(p.exists() for p in [t1, t1ce, t2, flair, seg]):
            train_files.append({
                "image": [str(t1), str(t1ce), str(t2), str(flair)],
                "label": str(seg)
            })

print("Total samples:", len(train_files))
print(train_files[0])

Total samples: 1350
{'image': ['C:\\Users\\Admin\\Desktop\\meghana\\Brats2024\\BraTS2024-BraTS-GLI-TrainingData\\training_data1_v2\\BraTS-GLI-00005-100\\BraTS-GLI-00005-100-t1n.nii.gz', 'C:\\Users\\Admin\\Desktop\\meghana\\Brats2024\\BraTS2024-BraTS-GLI-TrainingData\\training_data1_v2\\BraTS-GLI-00005-100\\BraTS-GLI-00005-100-t1c.nii.gz', 'C:\\Users\\Admin\\Desktop\\meghana\\Brats2024\\BraTS2024-BraTS-GLI-TrainingData\\training_data1_v2\\BraTS-GLI-00005-100\\BraTS-GLI-00005-100-t2w.nii.gz', 'C:\\Users\\Admin\\Desktop\\meghana\\Brats2024\\BraTS2024-BraTS-GLI-TrainingData\\training_data1_v2\\BraTS-GLI-00005-100\\BraTS-GLI-00005-100-t2f.nii.gz'], 'label': 'C:\\Users\\Admin\\Desktop\\meghana\\Brats2024\\BraTS2024-BraTS-GLI-TrainingData\\training_data1_v2\\BraTS-GLI-00005-100\\BraTS-GLI-00005-100-seg.nii.gz'}


In [3]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_files,
    test_size=0.2,
    random_state=42
)

print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))

Train samples: 1080
Validation samples: 270


In [4]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    EnsureTyped,
    MapTransform,
)
import torch


# FIX: Use torch instead of np.where — safe for MONAI MetaTensors
class ConvertLabels(MapTransform):
    """Remap BraTS label 4 → 3 (NCR=1, ED=2, ET=3, background=0)."""
    def __init__(self, keys):
        super().__init__(keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            lbl = d[key]
            # Convert to plain tensor if it's a MetaTensor, remap, restore
            d[key] = torch.where(
                torch.as_tensor(lbl) == 4,
                torch.tensor(3, dtype=lbl.dtype if hasattr(lbl, 'dtype') else torch.long),
                torch.as_tensor(lbl)
            )
        return d


train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    ConvertLabels(keys="label"),   # FIX: torch-safe version

    # Crop FIRST — major speed boost (operates on full vol only for pos/neg sampling)
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=ROI_SIZE,
        pos=1,
        neg=1,
        num_samples=1,
    ),

    # Normalize AFTER cropping (faster: smaller tensor)
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),

    EnsureTyped(keys=["image", "label"]),
])


val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    ConvertLabels(keys="label"),   # FIX: torch-safe version

    # No cropping for val — sliding window inference handles full volume
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),

    EnsureTyped(keys=["image", "label"]),
])

print("Transforms defined.")

Transforms defined.


In [5]:
from monai.data import Dataset, DataLoader

# No caching — straight disk reads, no deadlock risk on Windows
train_ds = Dataset(
    data=train_data,
    transform=train_transforms,
)

val_ds = Dataset(
    data=val_data,
    transform=val_transforms,
)

train_loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,       # 0 = main process only, no spawning, no deadlock
    pin_memory=False,
    persistent_workers=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)

print("Dataloader ready!")

Dataloader ready!


In [6]:
# # Sanity check batch shapes
# batch = next(iter(train_loader))
# print("Image shape:", batch["image"].shape)   # expect (4, 4, 128, 128, 128)
# print("Label shape:", batch["label"].shape)   # expect (4, 1, 128, 128, 128)
# print("Label unique values:", batch["label"].unique())  # should be 0,1,2,3 only

In [7]:
from monai.networks.nets import SwinUNETR

model = SwinUNETR(
    in_channels=4,
    out_channels=NUM_CLASSES,
    feature_size=48,
    use_checkpoint=True,     # FIX: gradient checkpointing — saves ~30% VRAM,
                             # critical when BATCH_SIZE=4
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded on: {DEVICE}")
print(f"Trainable parameters: {total_params / 1e6:.1f}M")

Model loaded on: cuda
Trainable parameters: 62.2M


In [8]:
from monai.losses import DiceCELoss
import torch

loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)

# FIX: Cosine annealing LR scheduler — flat LR for 20 epochs leaves perf on the table
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

print("Loss, optimizer and scheduler ready.")

Loss, optimizer and scheduler ready.


In [9]:
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete

# FIX: include_background=False — BraTS evaluates WT/TC/ET, not background.
# include_background=True inflates scores by counting the easy background class.
# reduction='mean_batch' gives per-class dice so you can track WT, TC, ET separately.
dice_metric = DiceMetric(include_background=False, reduction="mean_batch")

# Convert predictions → one-hot class labels (applied per sample after decollate)
post_pred  = AsDiscrete(argmax=True, to_onehot=NUM_CLASSES)

# Convert labels → one-hot (applied per sample after decollate)
post_label = AsDiscrete(to_onehot=NUM_CLASSES)

print("Validation setup ready.")
print("Dice metric will report 3 values: NCR (class1), ED (class2), ET (class3)")

Validation setup ready.
Dice metric will report 3 values: NCR (class1), ED (class2), ET (class3)


In [10]:
from torch.amp import autocast, GradScaler
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch
from tqdm import tqdm

scaler = GradScaler("cuda")

best_dice   = 0.0
train_losses = []
val_dices    = []

for epoch in range(MAX_EPOCHS):
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS}  |  LR: {scheduler.get_last_lr()[0]:.2e}")

    # ═══════════════════════ TRAIN ═══════════════════════
    model.train()
    epoch_loss = 0.0

    pbar = tqdm(train_loader, desc="Training")
    for batch in pbar:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        with autocast("cuda"):
            outputs = model(images)
            # FIX: pass labels directly (shape B,1,H,W,D)
            # DO NOT squeeze — DiceCELoss(to_onehot_y=True) expects channel dim=1
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_description(f"Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Train Loss: {avg_loss:.4f}")

    # Step LR scheduler after each epoch
    scheduler.step()

    # ═══════════════════════ VALIDATION ═══════════════════════
    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        dice_metric.reset()

        with torch.no_grad():
            for val_batch in tqdm(val_loader, desc="Validation"):
                val_images = val_batch["image"].to(DEVICE)
                val_labels = val_batch["label"].to(DEVICE)

                # FIX: sliding window inference — full BraTS volumes are 240×240×155.
                # Direct forward pass on full volume is wrong (model trained on 128³)
                # and will OOM. sliding_window_inference tiles the volume correctly.
                with autocast("cuda"):
                    val_outputs = sliding_window_inference(
                        val_images,
                        ROI_SIZE,
                        SW_BATCH_SIZE,
                        model,
                        overlap=0.5,
                    )

                # FIX: decollate_batch + apply post transforms per sample.
                # AsDiscrete operates on single samples, not batches.
                val_outputs_list = [post_pred(i)  for i in decollate_batch(val_outputs)]
                val_labels_list  = [post_label(i) for i in decollate_batch(val_labels)]

                dice_metric(y_pred=val_outputs_list, y=val_labels_list)

        # mean_batch returns per-class dice: [NCR, ED, ET]
        per_class_dice = dice_metric.aggregate()
        mean_dice = per_class_dice.mean().item()
        dice_metric.reset()

        val_dices.append(mean_dice)
        print(f"Validation Dice  —  NCR: {per_class_dice[0]:.4f} | "
              f"ED: {per_class_dice[1]:.4f} | ET: {per_class_dice[2]:.4f} | "
              f"Mean: {mean_dice:.4f}")

        # FIX: save best model checkpoint, not just final weights
        if mean_dice > best_dice:
            best_dice = mean_dice
            torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pth")
            print(f"  ✅ New best model saved (dice={best_dice:.4f})")

# ═══════════════════════ SAVE FINAL MODEL ═══════════════════════
torch.save(model.state_dict(), OUTPUT_DIR / "swinunetr_brats_final.pth")
print(f"\nTraining complete! Best val dice: {best_dice:.4f}")
print("Final model saved to outputs/swinunetr_brats_final.pth")
print("Best model saved to outputs/best_model.pth")


Epoch 1/20  |  LR: 1.00e-04


Loss: 0.4862: 100%|██████████| 540/540 [6:15:02<00:00, 41.67s/it]      


Train Loss: 0.8846

Epoch 2/20  |  LR: 9.94e-05


Loss: 0.4813: 100%|██████████| 540/540 [43:17<00:00,  4.81s/it]


Train Loss: 0.5452


...

Train Loss: 0.3122

Epoch 20/20  |  LR: 6.16e-07


Loss: 0.2626: 100%|██████████| 540/540 [42:44<00:00,  4.75s/it]


Train Loss: 0.3096


Validation: 100%|██████████| 270/270 [15:13<00:00,  3.38s/it]


Validation Dice  —  NCR: 0.6425 | ED: 0.8778 | ET: 0.8542 | Mean: 0.7915
  ✅ New best model saved (dice=0.7915)

Training complete! Best val dice: 0.7915
Final model saved to outputs/swinunetr_brats_final.pth
Best model saved to outputs/best_model.pth


In [11]:
# Training curve summary
print("\n── Training Loss per Epoch ──")
for i, loss in enumerate(train_losses, 1):
    print(f"  Epoch {i:>2}: {loss:.4f}")

print("\n── Validation Dice (every 2 epochs) ──")
for i, dice in enumerate(val_dices, 1):
    print(f"  Val {i:>2}: {dice:.4f}")

print(f"\nBest Mean Dice: {best_dice:.4f}")


── Training Loss per Epoch ──
  Epoch  1: 0.8846
  Epoch  2: 0.5452
  Epoch  3: 0.4598
  Epoch  4: 0.4240
  Epoch  5: 0.3994
  Epoch  6: 0.3859
  Epoch  7: 0.3742
  Epoch  8: 0.3656
  Epoch  9: 0.3530
  Epoch 10: 0.3488
  Epoch 11: 0.3390
  Epoch 12: 0.3364
  Epoch 13: 0.3324
  Epoch 14: 0.3259
  Epoch 15: 0.3192
  Epoch 16: 0.3168
  Epoch 17: 0.3136
  Epoch 18: 0.3174
  Epoch 19: 0.3122
  Epoch 20: 0.3096

── Validation Dice (every 2 epochs) ──
  Val  1: 0.6118
  Val  2: 0.6849
  Val  3: 0.7118
  Val  4: 0.7254
  Val  5: 0.7570
  Val  6: 0.7607
  Val  7: 0.7722
  Val  8: 0.7890
  Val  9: 0.7857
  Val 10: 0.7915

Best Mean Dice: 0.7915


In [12]:
import einops
print("einops version:", einops.__version__)

einops version: 0.8.2
